<a href="https://colab.research.google.com/github/AbdallaYoussef006/FlyRank-AI/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
con.sql(f"""
SELECT *
FROM {TABLES['fact_daily']}
LIMIT 5
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115,...,0,0,0,0,0,0,0,0,0,2025-01
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358,...,0,0,0,0,0,0,0,0,0,2025-01
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34,...,0,0,0,0,0,0,0,0,0,2025-01
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140,...,0,0,0,0,0,0,0,0,0,2025-01
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89,...,0,0,0,0,0,0,0,0,0,2025-01


In [ ]:
%pip -q install duckdb huggingface_hub

In [ ]:
import os
import duckdb

from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
TYPE huggingface,
TOKEN '{HF_TOKEN}'
)
""")

In [ ]:
REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients":
        f"read_parquet('{REL}/dim_clients.parquet')",

    "dim_content":
        f"read_parquet('{REL}/dim_content.parquet')",

    "fact_daily":
        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",

    "fact_daily_sample":
        f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",

    "fact_query_90d":
        f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## Unit of analysis

One row represents the daily search performance of one content page for one day.

## Time window

This notebook uses March 2026 (month = 2026-03), a mid-panel month, to avoid using the final month for model development and evaluation.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## Features
- gsc_impressions
- gsc_clicks
- gsc_avg_position
- visible_queries
- rare_impressions_share

These are used as model features because they describe search performance before the prediction.

## Label
Whether the content shows declining search performance.

## Context
- report_date
- client_hash_id
- content_hash_id

These identify the observation but are not prediction features.

## Excluded
- Future information that would reveal the answer.
- Final month (June 2026) to avoid data leakage.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT report_date || client_hash_id || content_hash_id) AS unique_rows
FROM {TABLES['fact_daily']}
LIMIT 1
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_rows
0,78835655,78829265


In [ ]:
con.sql(f"""
SELECT
    MIN(report_date) AS first_day,
    MAX(report_date) AS last_day
FROM {TABLES['fact_daily']}
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,first_day,last_day
0,2025-01-27,2026-06-30


In [ ]:
con.sql(f"""
SELECT
    SUM(CASE WHEN gsc_impressions IS NULL THEN 1 ELSE 0 END) AS missing_impressions,
    SUM(CASE WHEN gsc_clicks IS NULL THEN 1 ELSE 0 END) AS missing_clicks,
    SUM(CASE WHEN gsc_sum_position IS NULL THEN 1 ELSE 0 END) AS missing_position
FROM {TABLES['fact_daily']}
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,missing_impressions,missing_clicks,missing_position
0,98006.0,98006.0,98014.0


In [ ]:
con.sql(f"""
SELECT
    COUNT(*) AS march_rows
FROM {TABLES['fact_daily']}
WHERE month = '2026-03'
""").df()

,march_rows
0,9841378


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

- Search data shows what happened, not why it happened.
- Clients have different amounts of historical data.
- The chosen month may not represent all seasons.
- Future information cannot be used because it would leak the answer.
- The anonymized data limits business interpretation.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.